In [29]:
import pandas as pd
import numpy as np
import spiceypy as spy
import tqdm
from Utils import CanonicalUnits, GravitationalParameters

In [30]:
path_data = "../datos/sbdb_query_results_NEOS.csv"

neos = pd.read_csv(path_data)
neos["q"]=neos["a"]*(1-neos["e"])
neos = neos[neos["q"] < 1.30]
neos

,pdes,epoch_mjd,a,e,i,om,w,ma,q
0,433,60200,1.458,0.2228,10.83,304.29,178.91,222.76,1.133158
1,719,60200,2.636,0.5470,11.58,183.85,156.23,56.29,1.194108
2,887,60200,2.472,0.5709,9.40,110.42,350.47,238.76,1.060735
3,1036,60200,2.666,0.5329,26.69,215.50,132.48,276.42,1.245289
4,1221,60200,1.919,0.4357,11.88,171.32,26.65,123.53,1.082892
...,...,...,...,...,...,...,...,...,...
34323,2024 CP5,60200,2.588,0.6057,3.48,309.59,206.24,321.35,1.020448
34324,2024 CQ5,60350,2.180,0.8506,2.16,228.85,146.50,16.09,0.325692
34325,2024 CR5,60352,1.136,0.3680,17.82,141.95,100.20,302.45,0.717952
34326,2024 CS5,60353,1.121,0.4184,42.92,140.26,168.63,222.21,0.651974


In [31]:
orbital_elements = np.column_stack((np.array(neos['a']), np.array(neos['e']), np.array(neos['i']), np.array(neos['om']), np.array(neos['w']), np.array(neos['ma'])))
orbital_elements

array([[1.4580e+00, 2.2280e-01, 1.0830e+01, 3.0429e+02, 1.7891e+02,
        2.2276e+02],
       [2.6360e+00, 5.4700e-01, 1.1580e+01, 1.8385e+02, 1.5623e+02,
        5.6290e+01],
       [2.4720e+00, 5.7090e-01, 9.4000e+00, 1.1042e+02, 3.5047e+02,
        2.3876e+02],
       ...,
       [1.1360e+00, 3.6800e-01, 1.7820e+01, 1.4195e+02, 1.0020e+02,
        3.0245e+02],
       [1.1210e+00, 4.1840e-01, 4.2920e+01, 1.4026e+02, 1.6863e+02,
        2.2221e+02],
       [2.8210e+00, 6.6180e-01, 4.6800e+00, 1.8281e+02, 2.3497e+02,
        1.3025e+02]])

In [32]:
deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s
mu = CanonicalUnits().mu
grav_params = GravitationalParameters(mu=mu)

In [33]:
state_vectors = np.zeros((len(orbital_elements), 6))
for index, elements in enumerate(orbital_elements):
    a = elements[0]
    e = elements[1]
    i = elements[2]
    Omega = elements[3]
    w = elements[4]
    M = elements[5]
    q = a*(1-e)

    state_vector = spy.conics([q, e, i, Omega, w, M]+[0, mu], 0)
    state_vectors[index] = np.array(state_vector[:6])

In [34]:
state_vectors

array([[-1.54612062e+00,  6.24587517e-01,  6.03978253e-01,
        -1.22034861e+00,  1.28152560e+00, -3.78361274e+00],
       [ 7.71717261e-01, -2.59916862e-01,  1.13775254e+00,
        -7.81595664e-01,  6.36936789e+00, -5.35899844e-01],
       [ 2.99600366e-01, -1.01722099e+00, -2.58660844e-02,
        -7.33906247e+00, -2.14896049e+00,  3.38195796e-02],
       ...,
       [ 8.27615153e-05, -5.74118359e-01, -7.99841623e-01,
         5.58844230e+00,  1.04706964e+00, -3.62325617e+00],
       [-5.42635247e-01, -5.36476980e-01, -1.29924829e+00,
         1.65543610e+00, -3.78528642e+00, -3.47171696e-01],
       [ 3.18154313e+00,  2.19788310e+00,  8.25187233e-01,
        -7.48707030e-01, -5.97761677e-01, -2.24905792e+00]])

In [49]:
import plotly.graph_objects as go

import numpy as np

xs = state_vectors[:,0]
xs = xs[xs > -40]
ys = state_vectors[:,1]
zs = state_vectors[:,2]
xs[(xs < 15) & (xs > -40)]
# Add 1 to shift the mean of the Gaussian distribution

fig = go.Figure()
fig.add_trace(go.Histogram(x=xs))
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.show()

In [50]:
len(xs[xs > 10]), len(xs)

(1, 34322)

In [37]:
np.max(xs), np.min(xs), np.max(ys), np.min(ys), np.max(zs), np.min(zs)

(33.81022229578148,
 -229.6877518555222,
 188.60283966378051,
 -10.245424703541705,
 164.18667964710014,
 -136.48349916678754)

In [ ]:
import plotly.graph_objects as go

import numpy as np
x0 = np.random.randn(500)
x1 = np.random.randn(500) + 1

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=x0,
    histnorm='percent',
    name='control', # name used in legend and hover labels
    xbins=dict( # bins used for histogram
        start=-4.0,
        end=3.0,
        size=0.5
    ),
    marker_color='#EB89B5',
    opacity=0.75
))
fig.add_trace(go.Histogram(
    x=x1,
    histnorm='percent',
    name='experimental',
    xbins=dict(
        start=-3.0,
        end=4,
        size=0.5
    ),
    marker_color='#330C73',
    opacity=0.75
))

fig.update_layout(
    title_text='Sampled Results', # title of plot
    xaxis_title_text='Value', # xaxis label
    yaxis_title_text='Count', # yaxis label
    bargap=0.2, # gap between bars of adjacent location coordinates
    bargroupgap=0.1 # gap between bars of the same location coordinates
)

fig.show()